# SWAT+ Output Aggregation — Full Scenario Matrix per Channel

Extracts long-term monthly average flow for one or more channels across the
full LU (Land Use) x CC (Climate Change) scenario matrix produced by SWAT+
(via `9_data_extraction_from_swat_output.ipynb`).

**Directory logic:**
```
SWAT_Outputs/
  2010_lulc_Historical BASELINE model/      <- baseline LULC (2010)
      historical_climate/Channel_combined/channel_longterm_monthly_avg.csv
          -> Scenario_Type = 'Historical'            (1 run)
      {8 climate scenarios}/Channel_combined/{near|mid|far}_future/...csv
          -> Scenario_Type = 'CC_Only'                (8 x 3 = 24 runs)

  2040_lulc_model/   (-> near_future)
  2065_lulc_model/   (-> mid_future)
  2090_lulc_model/   (-> far_future)
      historical_climate/Channel_combined/channel_longterm_monthly_avg.csv
          -> Scenario_Type = 'LU_Only'                (3 runs, one per LULC year)
      {8 climate scenarios}/Channel_combined/channel_longterm_monthly_avg.csv
          -> Scenario_Type = 'LU_CC'                  (3 x 8 = 24 runs)

Total = 1 + 3 + 24 + 24 = 52 monthly-average series per channel.
```

**Writes:** one wide-format (52 columns) and one tidy long-format CSV per
channel, to `SWAT_Outputs/aggregated_results/flow_{channel}/` — the direct
input to `11_Unceratinty_analysis.ipynb` / `uq_toolkit.py`.

## Config

In [ ]:
import os

import pandas as pd

BASE_DIR = '../SWATPlus Models/SWAT_Outputs'

LULC_PERIOD_MAP = {
    '2040_lulc_model': 'near_future',
    '2065_lulc_model': 'mid_future',
    '2090_lulc_model': 'far_future',
}
BASELINE_LULC_NAME = '2010_lulc_Historical BASELINE model'

CC_SCENARIOS = [
    '245_Cool-Dry', '245_Cool-Wet', '245_Hot-Dry', '245_Hot-Wet',
    '585_Cool-Dry', '585_Cool-Wet', '585_Hot-Dry', '585_Hot-Wet',
]

FUTURE_PERIODS = ['near_future', 'mid_future', 'far_future']

MONTH_ORDER = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']


## Core functions — build the tidy dataset, then export wide & long CSVs

In [ ]:

def _load_channel_series(csv_path, channel_col):
    df = pd.read_csv(csv_path)
    if channel_col not in df.columns:
        raise KeyError(
            f"Channel '{channel_col}' not found in {csv_path}. "
            f"Available columns: {[c for c in df.columns if c != 'DateMonth']}"
        )
    return df.set_index('DateMonth')[channel_col]


def build_channel_dataset(channel_col, base_dir=BASE_DIR, verbose=True):
    """
    Walk the full SWAT_Outputs tree once and return a tidy long-format
    DataFrame: Scenario_Type, LULC_Year, Climate_Scenario, Future_Period,
    Month, Flow -- one row per (scenario combination, month).
    """
    records = []
    missing = []

    all_lulcs = [d for d in os.listdir(base_dir) if d.endswith('model')]

    for lulc in all_lulcs:
        lulc_path = os.path.join(base_dir, lulc)
        is_baseline = (lulc == BASELINE_LULC_NAME)
        lulc_year = lulc.split('_')[0]

        hist_csv = os.path.join(lulc_path, 'historical_climate',
                                 'Channel_combined',
                                 'channel_longterm_monthly_avg.csv')
        if os.path.exists(hist_csv):
            series = _load_channel_series(hist_csv, channel_col)
            scenario_type = 'Historical' if is_baseline else 'LU_Only'
            future_period = None if is_baseline else LULC_PERIOD_MAP[lulc]
            for month, val in series.items():
                records.append({
                    'Scenario_Type': scenario_type,
                    'LULC_Year': lulc_year,
                    'Climate_Scenario': 'historical',
                    'Future_Period': future_period,
                    'Month': month,
                    'Flow': val,
                })
        else:
            missing.append(hist_csv)

        for scen in CC_SCENARIOS:
            scen_path = os.path.join(lulc_path, scen, 'Channel_combined')
            if not os.path.isdir(scen_path):
                missing.append(scen_path)
                continue

            if is_baseline:
                for period in FUTURE_PERIODS:
                    csv_path = os.path.join(
                        scen_path, period, 'channel_longterm_monthly_avg.csv')
                    if os.path.exists(csv_path):
                        series = _load_channel_series(csv_path, channel_col)
                        for month, val in series.items():
                            records.append({
                                'Scenario_Type': 'CC_Only',
                                'LULC_Year': lulc_year,
                                'Climate_Scenario': scen,
                                'Future_Period': period,
                                'Month': month,
                                'Flow': val,
                            })
                    else:
                        missing.append(csv_path)
            else:
                csv_path = os.path.join(
                    scen_path, 'channel_longterm_monthly_avg.csv')
                if os.path.exists(csv_path):
                    series = _load_channel_series(csv_path, channel_col)
                    for month, val in series.items():
                        records.append({
                            'Scenario_Type': 'LU_CC',
                            'LULC_Year': lulc_year,
                            'Climate_Scenario': scen,
                            'Future_Period': LULC_PERIOD_MAP[lulc],
                            'Month': month,
                            'Flow': val,
                        })
                else:
                    missing.append(csv_path)

    df = pd.DataFrame(records)
    df['Month'] = pd.Categorical(df['Month'], categories=MONTH_ORDER, ordered=True)
    df = df.sort_values(
        ['Scenario_Type', 'LULC_Year', 'Future_Period', 'Climate_Scenario', 'Month']
    ).reset_index(drop=True)

    n_series = df.drop_duplicates(
        ['Scenario_Type', 'LULC_Year', 'Climate_Scenario', 'Future_Period']
    ).shape[0]

    if verbose:
        print(f"Channel '{channel_col}': found {n_series} monthly-average series "
              f"({len(df)} rows). Expected 52 series.")
        if missing:
            print(f"  {len(missing)} expected paths were missing, e.g.:")
            for m in missing[:5]:
                print(f"    - {m}")

    return df


def export_wide_csv(df, out_path):
    """
    Write one CSV: 52 columns (one per scenario), first column holds row
    labels. Row layout:
        Scenario_Type   Historical  LU_Only  LU_Only  LU_Only  CC_Only ...
        Period          historical  near_..  mid_..   far_..   near_.. ...
        Emission        historical  historical ...             245    ...
        Climate_Model   historical  historical ...             Cool-Dry ...
        Jan             <values>
        Feb             <values>
        ...
        Dec             <values>
    """
    work = df.copy()

    def split_emission_model(cs):
        if cs == 'historical':
            return pd.Series(['historical', 'historical'])
        emission, model = cs.split('_', 1)
        return pd.Series([emission, model])

    work[['Emission', 'Climate_Model']] = work['Climate_Scenario'].apply(split_emission_model)
    work['Period'] = work.apply(
        lambda r: 'historical' if r['Scenario_Type'] == 'Historical' else r['Future_Period'],
        axis=1
    )

    scenario_order = ['Historical', 'LU_Only', 'CC_Only', 'LU_CC']
    period_order = ['historical', 'near_future', 'mid_future', 'far_future']
    emission_order = ['historical', '245', '585']
    model_order = ['historical', 'Cool-Dry', 'Cool-Wet', 'Hot-Dry', 'Hot-Wet']

    combos = (work[['Scenario_Type', 'Period', 'Emission', 'Climate_Model']]
              .drop_duplicates())
    combos['_s'] = combos['Scenario_Type'].map({v: i for i, v in enumerate(scenario_order)})
    combos['_p'] = combos['Period'].map({v: i for i, v in enumerate(period_order)})
    combos['_e'] = combos['Emission'].map({v: i for i, v in enumerate(emission_order)})
    combos['_m'] = combos['Climate_Model'].map({v: i for i, v in enumerate(model_order)})
    combos = combos.sort_values(['_s', '_p', '_e', '_m']).drop(columns=['_s', '_p', '_e', '_m'])

    n = len(combos)
    header_rows = pd.DataFrame({
        i: [row.Scenario_Type, row.Period, row.Emission, row.Climate_Model]
        for i, row in enumerate(combos.itertuples())
    }, index=['Scenario_Type', 'Period', 'Emission', 'Climate_Model'])

    month_pivot = work.pivot_table(index='Month', columns='Climate_Scenario', values='Flow')
    # need to pivot on the FULL combo (Scenario_Type, Period, Emission, Climate_Model),
    # not just Climate_Scenario (which repeats across Scenario_Type/Period)
    work['_col_key'] = list(zip(work.Scenario_Type, work.Period, work.Emission, work.Climate_Model))
    combos['_col_key'] = list(zip(combos.Scenario_Type, combos.Period, combos.Emission, combos.Climate_Model))
    key_to_i = {k: i for i, k in enumerate(combos['_col_key'])}
    work['_col_i'] = work['_col_key'].map(key_to_i)

    data_rows = work.pivot_table(index='Month', columns='_col_i', values='Flow', observed=True)
    data_rows = data_rows.reindex(index=MONTH_ORDER, columns=range(n))

    combined = pd.concat([header_rows, data_rows])
    combined.to_csv(out_path, header=False)
    print(f"Saved: {out_path} ({n} scenario columns)")


SCENARIO_LABEL = {'Historical': 'Historical', 'LU_Only': 'LU only',
                   'CC_Only': 'CC only', 'LU_CC': 'LU+CC'}
TYPE_LABEL = {'Historical': 'Baseline', 'LU_Only': 'LU',
              'CC_Only': 'CC', 'LU_CC': 'Combined'}
PERIOD_LABEL = {'near_future': 'Near', 'mid_future': 'Mid', 'far_future': 'Far'}

SCENARIO_ORDER = ['Historical', 'LU_Only', 'CC_Only', 'LU_CC']


def build_long_csv(df):
    """
    Tidy long-format table:
        Month  Scenario  Period  SSP  Climate  Type  Flow
    - Scenario: Historical / LU only / CC only / LU+CC
    - Period:   Historical / Near / Mid / Far
    - SSP:      '-' for Historical & LU only, else 245 / 585
    - Climate:  '-' for Historical & LU only, else Cool-Dry/Cool-Wet/Hot-Dry/Hot-Wet
    - Type:     Baseline / LU / CC / Combined
    """
    work = df.copy()
    work['Scenario'] = work['Scenario_Type'].map(SCENARIO_LABEL)
    work['Type'] = work['Scenario_Type'].map(TYPE_LABEL)
    work['Period'] = work['Future_Period'].map(PERIOD_LABEL).fillna('Historical')

    def split_ssp_climate(row):
        if row['Scenario_Type'] in ('Historical', 'LU_Only'):
            return pd.Series(['-', '-'])
        emission, model = row['Climate_Scenario'].split('_', 1)
        return pd.Series([emission, model])

    work[['SSP', 'Climate']] = work.apply(split_ssp_climate, axis=1)

    work['_s'] = work['Scenario_Type'].map({v: i for i, v in enumerate(SCENARIO_ORDER)})
    period_sort_order = {'near_future': 0, 'mid_future': 1, 'far_future': 2}
    work['_p'] = work['Future_Period'].map(period_sort_order).fillna(-1)
    work = work.sort_values(['Month', '_s', '_p', 'Climate_Scenario'])

    return work[['Month', 'Scenario', 'Period', 'SSP', 'Climate', 'Type', 'Flow']].reset_index(drop=True)


def export_long_csv(df, out_path):
    long_df = build_long_csv(df)
    long_df.to_csv(out_path, index=False)
    print(f"Saved: {out_path} ({len(long_df)} rows)")

## Run for each channel

In [ ]:
CHANNELS_TO_EXTRACT = [53, 98, 69, 50]

for channel in CHANNELS_TO_EXTRACT:
    CHANNEL = f'cha{channel}_flo_out'
    data = build_channel_dataset(CHANNEL)
    output_dir = f'../SWATPlus Models/SWAT_Outputs/aggregated_results/flow_{channel}/'
    os.makedirs(output_dir, exist_ok=True)
    export_wide_csv(data, f'{output_dir}{CHANNEL}_wide52.csv')
    export_long_csv(data, f'{output_dir}{CHANNEL}_long.csv')
